In [1]:
import pandas as pd
import geopandas as gpd
import plotly.express as px

In [16]:
## collecting all of the neccesary data
df_Ageo_top20 = pd.read_csv("datasets_rq4/df_Ageo_top20.csv")
df_AID_TOP20_YMA = pd.read_csv("datasets_rq4/df_AID_TOP20_YMA.csv")

world = gpd.read_file("https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip")

df_Ageo_top20.head()
df_AID_TOP20_YMA.head()

,Unnamed: 0,Year_Lobt,Month_Lobt,APT_ICAO,Total Delay (TD),Total Flights (TF),Total_Flights_Period,Avg Delay per Movement (in min),Avg Proportion of Delay (%),Delay_Ratio
0,0,2024,July,EDDF,295785,25288,244460,1.209953,4.243149,10.344433
1,1,2024,August,EDDF,242797,23770,244137,0.994511,4.492451,9.736337
2,2,2024,September,EDDF,213238,23435,241043,0.884647,4.513545,9.722332
3,3,2023,August,EDDF,291567,21990,240295,1.213371,4.496111,9.151252
4,4,2023,July,EDDF,246893,21909,237439,1.039817,4.288736,9.227212


Merging all datasets to one

In [32]:
# changing month column to numbers
df_AID_TOP20_YMA['Month_Lobt_num'] = pd.to_datetime(df_AID_TOP20_YMA['Month_Lobt'], format='%B').dt.month

# merging the year and month column
df_AID_TOP20_YMA['month_year'] = (
    df_AID_TOP20_YMA['Year_Lobt'].astype(str) + '-' +
    df_AID_TOP20_YMA['Month_Lobt_num'].astype(str).str.zfill(2)
)
# merging df_Ageo and df_AT
df_total1 = pd.merge(df_AID_TOP20_YMA, df_Ageo_top20,
                    left_on='APT_ICAO', right_on='ident', how='left')

# Sorting by date
df_total1 = df_total1.sort_values('month_year').reset_index(drop=True)
df_total1


,Unnamed: 0_x,Year_Lobt,Month_Lobt,APT_ICAO,Total Delay (TD),Total Flights (TF),Total_Flights_Period,Avg Delay per Movement (in min),Avg Proportion of Delay (%),Delay_Ratio,month_year,Month_Lobt_num,Unnamed: 0_y,ident,name,latitude_deg,longitude_deg,elevation_ft
0,419,2023,January,LGAV,24117,2342,71264,0.338418,4.155381,3.286372,2023-01,1,12,LGAV,Eleftherios Venizelos International Airport,37.936401,23.944500,308.0
1,359,2023,January,EIDW,44231,3450,108698,0.406916,3.695142,3.173931,2023-01,1,5,EIDW,Dublin Airport,53.421299,-6.270070,242.0
2,343,2023,January,LOWW,39008,3608,105383,0.370155,3.680115,3.423702,2023-01,1,14,LOWW,Vienna International Airport,48.110298,16.569700,600.0
3,451,2023,January,LTFJ,24794,1816,48671,0.509420,4.610890,3.731175,2023-01,1,18,LTFJ,Istanbul Sabiha Gökçen International Airport,40.898602,29.309200,312.0
4,338,2023,January,LIRF,44565,3714,115957,0.384323,3.486786,3.202911,2023-01,1,13,LIRF,Leonardo da Vinci–Fiumicino Airport,41.800278,12.238889,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
474,231,2024,December,LIRF,69884,5991,149464,0.467564,3.334028,4.008323,2024-12,12,13,LIRF,Leonardo da Vinci–Fiumicino Airport,41.800278,12.238889,13.0
475,139,2024,December,LTFM,172115,8441,268141,0.641882,4.753089,3.147971,2024-12,12,19,LTFM,Istanbul Airport,41.261297,28.741951,325.0
476,190,2024,December,LSZH,70419,7095,111418,0.632025,4.024693,6.367912,2024-12,12,16,LSZH,Zürich Airport,47.464699,8.549170,1416.0
477,460,2024,December,LEPA,15326,1424,55114,0.278078,2.777612,2.583736,2024-12,12,10,LEPA,Palma De Mallorca Airport,39.551701,2.738810,27.0


## Code for the map

In [40]:
# Min and Max delay, so the colorscheme stays consistent
min_delay = df_total1['Total Delay (TD)'].min()
max_delay = df_total1['Total Delay (TD)'].max()

# Code for the map
map_fig = px.scatter_map(
    df_total1,
    lat="latitude_deg",
    lon="longitude_deg",
    size="Total Flights (TF)",
    color="Total Delay (TD)",
    color_continuous_scale=["green", "yellow", "red"],
    range_color=[min_delay, max_delay],
    map_style="carto-positron",
    zoom=4,
    width=1000,
    height=700,
    animation_frame="month_year",
    size_max=50,
    hover_name="name", 
    hover_data={
        "Total Delay (TD)": True,
        "Total Flights (TF)": True,
        "Total_Flights_Period": True,
        "Avg Delay per Movement (in min)": True,
        "Avg Proportion of Delay (%)": True,
        "Delay_Ratio": True
        },
)

# Chaning the hover text
hover_text=("<b>%{hovertext}</b><br>" +
    "Total Delay: %{customdata[0]} minutes <br>" +
    "Total Flights: %{customdata[1]}<br>" +
    "Total Flights Period: %{customdata[2]}<br>" +
    "Average delay per Movement: %{customdata[3]:.2f} minutes <br>" +
    "Average Proportion of Delay: %{customdata[4]:.2f} % <br>" +
    "Delay ratio: %{customdata[5]:.2f} % <br>"	)

map_fig.update_traces(hovertemplate=hover_text)

# Ensuring  hover_text is applied for all animation frames
for frame in map_fig.frames:
    for trace in frame.data:
        trace.hovertemplate = hover_text

map_fig.show()
